# Phase A — Deterministic Work-Package Builder (IFRS S1/S2 Report Pipeline)

This notebook is **Phase A** of the report-generation pipeline. It contains **no LLM calls**. It converts the raw inputs (requirement KBs, bank payload slices, style system) into validated, self-contained **work packages** — one per report section — that the LangGraph agents in Phase B consume.

What it produces per section (`work_package_<section>.json`):

1. **Enriched requirements** — every requirement mapped to the payload keys that answer it (`mapped_payload_keys`), tagged with an `availability_status` (`data_backed` / `narrative_only` / `presentation_rule` / `not_applicable_archetype`) and a `handling_hint` (`disclosure` / `drafting_constraint` / `fixed_block` / `conditional_event`).
2. **Computed metrics** — every number the drafter is allowed to state, pre-computed here with provenance, canonical display formatting, and gap-aware caveats. The LLM never does arithmetic.
3. **Gap directives** — `metadata.data_gaps` compiled into enforceable rules: forbidden claims + the exact disclosure posture that keeps the report compliant without meta-commentary. Requirements satisfied via a gap directive score as **satisfied**, never penalized.
4. **Shared artifacts** — `allowed_numbers.json` (the closed set for the numeric hard gate), `entity_allowlist.json` (named entities the fact judge accepts), `run_manifest.json` (input hashes for auditability).

**Hard rule:** Phase B must not start if this notebook's final assertions fail.

In [12]:
# ============================================================
# CELL 1 — SETUP & CONFIG
# Notebook location : /notebooks
# Inputs            : requirement KBs, payload slices, style system
# Outputs           : /notebooks/gen_data/generation/phase_a/
# ============================================================

import hashlib
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd()

# ------------------------------------------------------------
# Configure these three input locations for your environment.
# Defaults assume all Phase A inputs sit in one flat folder;
# point them at your real export folders if they differ.
# ------------------------------------------------------------
REQUIREMENTS_DIR = NOTEBOOK_DIR / "gen_data" / "IFRS" / "ifrs_requirements_kb_outputs_final" / "section_by_section_requirements" / "json"  # *_requirements.json
PAYLOAD_DIR      = NOTEBOOK_DIR / "gen_data" / "UPLOADS" / "payloads"       # payload_BANK01*.json
STYLE_SYSTEM_DIR = NOTEBOOK_DIR / "gen_data" / "UPLOADS" / "style"   # style_system_*.json

OUTPUT_DIR = NOTEBOOK_DIR / "gen_data" / "generation" / "phase_a"
FORCE_RERUN = True

# Fallback: if the configured dirs don't exist but a single flat folder holds
# everything (e.g. during development), set FLAT_INPUT_DIR and both dirs collapse.
FLAT_INPUT_DIR = None  # e.g. Path("/mnt/project")
if FLAT_INPUT_DIR is not None:
    REQUIREMENTS_DIR = PAYLOAD_DIR = Path(FLAT_INPUT_DIR)

print("Requirements dir :", REQUIREMENTS_DIR)
print("Payload dir      :", PAYLOAD_DIR)
print("Style system dir :", STYLE_SYSTEM_DIR)
print("Output dir       :", OUTPUT_DIR)

Requirements dir : c:\Users\HP\Documents\IFRS_Reporting\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\section_by_section_requirements\json
Payload dir      : c:\Users\HP\Documents\IFRS_Reporting\notebooks\gen_data\UPLOADS\payloads
Style system dir : c:\Users\HP\Documents\IFRS_Reporting\notebooks\gen_data\UPLOADS\style
Output dir       : c:\Users\HP\Documents\IFRS_Reporting\notebooks\gen_data\generation\phase_a


## Cell 2 — Section registry, tag→payload map, and handling rules

The backbone of anti-hallucination: `TAG_TO_PAYLOAD_KEYS` maps every `evidence_tag` to the payload keys that can support it; `PARAGRAPH_OVERRIDES` sharpens banking-critical paragraphs (S2.29A financed emissions, B62/B62A disaggregation, S1.20/23); `SECTION_DEFAULT_KEYS` + `PARAGRAPH_HANDLING` cover framing paragraphs that have no evidence tags (fair presentation, statement of compliance, avoid-duplication, etc.) — those are satisfied by report behaviour or fixed blocks, not by data prose.

In [13]:
SECTION_REGISTRY = {
    "general_requirements": {
        "title": "General Requirements",
        "requirements_file": "general_requirements_requirements.json",
        "payload_file": "payload_BANK01_general_requirements.json",
    },
    "governance": {
        "title": "Governance",
        "requirements_file": "governance_requirements.json",
        "payload_file": "payload_BANK01_governance.json",
    },
    "strategy": {
        "title": "Strategy",
        "requirements_file": "strategy_requirements.json",
        "payload_file": "payload_BANK01_strategy.json",
    },
    "risk_management": {
        "title": "Risk Management",
        "requirements_file": "risk_management_requirements.json",
        "payload_file": "payload_BANK01_risk_management.json",
    },
    "metrics_and_targets": {
        "title": "Metrics and Targets",
        "requirements_file": "metrics_and_targets_requirements.json",
        "payload_file": "payload_BANK01_metrics_targets.json",
    },
}

REQUIRED_REQ_FIELDS = [
    "requirement_id", "standard", "paragraph_id", "report_section",
    "requirement_text", "clause_path", "obligation_type", "mandatory",
    "evidence_tags", "banking_relevance",
]

# ------------------------------------------------------------
# Evidence tag -> payload top-level keys (the requirement->data backbone).
# Keys are intersected with each section's payload slice at build time.
# ------------------------------------------------------------

TAG_TO_PAYLOAD_KEYS = {
    "governance_body": ["governance", "board_minutes", "reporting_kpis.governance_maturity"],
    "management_role": ["governance", "board_minutes"],
    "remuneration": ["governance"],
    "risk_process": ["climate_risk_register", "physical_risk_exposures", "governance",
                     "metadata.risk_rating_methodology"],
    "scenario_analysis": ["climate_scenarios", "resilience_assessment"],
    "business_model_value_chain": ["value_chain_map", "bank", "financial_summary"],
    "strategy_decision_making": ["transition_plan", "climate_opportunities",
                                 "internal_carbon_price", "targets"],
    "financial_effects": ["climate_financial_effects", "financial_summary", "reporting_kpis"],
    "materiality": ["general_requirements_context", "value_chain_map"],
    "connected_information": ["general_requirements_context"],
    "source_guidance": ["general_requirements_context"],
    "metrics": ["reporting_kpis", "scope1", "scope2", "scope3_travel",
                "financed_emissions", "financial_summary"],
    "targets": ["targets", "reporting_kpis.target_summary", "carbon_credits"],
    "ghg_emissions": ["scope1", "scope2", "scope3_travel", "scope3_categories",
                      "ghg_methodology", "scope12_consolidation", "financed_emissions"],
    "scope_1": ["scope1", "ghg_methodology", "metadata.vehicles_correction"],
    "scope_2": ["scope2", "ghg_methodology", "metadata.scope2_rec_reconciliation"],
    "scope_3": ["scope3_travel", "scope3_categories", "financed_emissions"],
    "financed_emissions": ["financed_emissions", "financed_emissions_equity",
                           "financed_emissions_sovereign", "metadata.pcaf_methodology"],
    "commercial_banking": ["financed_emissions", "financed_emissions_equity",
                           "financed_emissions_sovereign", "reporting_kpis"],
    "asset_management": ["financed_emissions_equity"],
    "insurance": [],  # bank archetype has no insurance activities -> conditional N/A
    "carbon_credits": ["carbon_credits", "targets"],
}

# Paragraph-level overrides: sharper mapping for banking-critical paragraphs.
PARAGRAPH_OVERRIDES = {
    ("IFRS S2", "29A"): ["financed_emissions", "financed_emissions_equity",
                         "financed_emissions_sovereign", "metadata.pcaf_methodology",
                         "metadata.data_gaps"],
    ("IFRS S2", "B62"): ["financed_emissions", "financed_emissions_equity",
                         "financed_emissions_sovereign", "metadata.pcaf_methodology"],
    ("IFRS S2", "B62A"): ["financed_emissions_equity", "financed_emissions_sovereign",
                          "metadata.pcaf_methodology"],
    ("IFRS S1", "20"): ["general_requirements_context", "bank"],
    ("IFRS S1", "23"): ["general_requirements_context", "metadata.data_gaps"],
}


# Fallback payload keys for requirements with NO evidence tags (framing/presentation
# paragraphs). Keys are per-section because payload slices differ.
SECTION_DEFAULT_KEYS = {
    "general_requirements": ["general_requirements_context", "metadata.data_gaps", "bank"],
    "governance": ["governance", "board_minutes"],
    "strategy": ["climate_risk_register", "climate_opportunities", "value_chain_map",
                 "climate_scenarios", "climate_financial_effects"],
    "risk_management": ["climate_risk_register", "metadata.risk_rating_methodology", "governance"],
    "metrics_and_targets": ["reporting_kpis", "targets", "ghg_methodology"],
}

# Paragraphs satisfied by PIPELINE BEHAVIOUR rather than generated prose.
# drafting_constraint -> enforced by judges / cross-section pass (e.g. avoid duplication)
# fixed_block        -> deterministic boilerplate block (e.g. statement of compliance)
# conditional_event  -> only applies if the event occurred (period change, error restatement)
PARAGRAPH_HANDLING = {
    ("IFRS S2", "7"): "drafting_constraint",     # avoid duplication with S1
    ("IFRS S2", "26"): "drafting_constraint",    # avoid duplication with S1
    ("IFRS S1", "B37"): "drafting_constraint",   # sensitive-info exemption limits
    ("IFRS S1", "72"): "fixed_block",            # statement of compliance
    ("IFRS S1", "73"): "fixed_block",
    ("IFRS S1", "74"): "fixed_block",
    ("IFRS S1", "66"): "conditional_event",      # change of reporting period end
    ("IFRS S1", "67"): "conditional_event",
    ("IFRS S1", "68"): "conditional_event",
    ("IFRS S1", "69"): "conditional_event",
    ("IFRS S1", "B59"): "conditional_event",     # error restatement impracticable
}

# Tags whose requirements are inherently narrative/process (no numeric payload needed).
NARRATIVE_OK_TAGS = {"connected_information", "source_guidance", "materiality"}

# Archetype-conditional tags: not applicable for a bank without those business lines.
CONDITIONAL_TAGS = {"insurance": "no_insurance_activities", "asset_management": "check_archetype"}

## Cell 3 — Generic helpers

In [14]:
# ------------------------------------------------------------
# Generic helpers
# ------------------------------------------------------------

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_by_path(obj, dotted: str):
    """Resolve 'metadata.pcaf_methodology' style paths against a dict."""
    cur = obj
    for part in dotted.split("."):
        if isinstance(cur, dict) and part in cur:
            cur = cur[part]
        else:
            return None
    return cur


def fmt_num(value, decimals=1):
    """Canonical display formatting: thousands separators, fixed decimals."""
    if value is None:
        return None
    if decimals == 0:
        return f"{value:,.0f}"
    return f"{value:,.{decimals}f}"

## Cell 4 — Load & validate requirement KBs

Checks required fields and global uniqueness of `requirement_id` across all five sections.

In [15]:
# ------------------------------------------------------------
# Step 1 - Load and validate requirement KBs
# ------------------------------------------------------------

def load_requirements(base_dir: Path):
    kbs, problems = {}, []
    seen_ids = set()
    for key, cfg in SECTION_REGISTRY.items():
        path = base_dir / cfg["requirements_file"]
        if not path.exists():
            problems.append(f"[{key}] requirements file missing: {path}")
            continue
        kb = load_json(path)
        reqs = []
        for std, blk in kb.get("standards", {}).items():
            for r in blk.get("requirements", []):
                missing = [f for f in REQUIRED_REQ_FIELDS if f not in r]
                if missing:
                    problems.append(f"[{key}] {r.get('requirement_id','?')} missing fields {missing}")
                rid = r.get("requirement_id")
                if rid in seen_ids:
                    problems.append(f"[{key}] duplicate requirement_id {rid}")
                seen_ids.add(rid)
                reqs.append(r)
        kbs[key] = {"section_title": kb.get("section_title"), "requirements": reqs,
                    "row_count": len(reqs), "source_file": str(path)}
    return kbs, problems

## Cell 5 — Load & validate payloads

Loads the full payload plus the five section slices, verifies `bank_id`/`reporting_year` consistency, and cross-checks the headline `reporting_kpis` figures against their source tables (Scope 1/2/3, financed emissions). A mismatch here means the payload build is broken — fail fast.

In [16]:
# ------------------------------------------------------------
# Step 2 - Load and validate payload slices + full payload
# ------------------------------------------------------------

def load_payloads(base_dir: Path, full_payload_file="payload_BANK01.json"):
    problems = []
    full = load_json(base_dir / full_payload_file)
    slices = {}
    bank_id = full["metadata"]["bank_id"]
    ryear = full["metadata"]["reporting_year"]
    for key, cfg in SECTION_REGISTRY.items():
        p = base_dir / cfg["payload_file"]
        sl = load_json(p)
        if sl["metadata"]["bank_id"] != bank_id:
            problems.append(f"[{key}] bank_id mismatch")
        if sl["metadata"]["reporting_year"] != ryear:
            problems.append(f"[{key}] reporting_year mismatch")
        slices[key] = sl

    # Cross-consistency: KPI headline figures must equal source tables (2024).
    kpi = full["reporting_kpis"]
    def row_for(table, year):
        return next((r for r in full[table] if r.get("reporting_year") == year), None)

    checks = [
        ("scope1_2024_tco2e", row_for("scope1", ryear), "scope1_total_tco2e"),
        ("scope2_location_2024_tco2e", row_for("scope2", ryear), "scope2_location_tco2e"),
        ("scope2_market_2024_tco2e", row_for("scope2", ryear), "scope2_market_tco2e"),
        ("scope3_travel_2024_tco2e", row_for("scope3_travel", ryear), "scope3_travel_tco2e"),
        ("financed_emissions_2024_tco2e", row_for("financed_emissions", ryear), "financed_em_loans_tco2e"),
    ]
    for kpi_key, row, col in checks:
        if row is None:
            problems.append(f"KPI check: no {ryear} row for {kpi_key}")
            continue
        if abs(kpi[kpi_key] - row[col]) > max(1e-6, abs(row[col]) * 1e-6):
            problems.append(f"KPI mismatch {kpi_key}: kpi={kpi[kpi_key]} table={row[col]}")
    return full, slices, problems

## Cell 6 — Gap directive compiler

Each `metadata.data_gaps` entry becomes a structured directive: affected field/years, **forbidden claims** (hard-gate fodder for the fact judge), and the **required disclosure posture** — the professional IFRS language that handles the gap without breaking the fourth wall. Judges receive these so gap-following text is scored as compliant, never as missing content.

In [17]:
# ------------------------------------------------------------
# Step 3 - Gap directive compiler
# ------------------------------------------------------------

GAP_FIELD_TO_TAGS = {
    "scope1_fleet_tco2e": ["scope_1", "ghg_emissions", "metrics"],
    "scope3_travel_tco2e": ["scope_3", "ghg_emissions", "metrics"],
    "investments.counterparty_id": ["financed_emissions", "commercial_banking", "asset_management"],
    "total_loans_meur": ["financed_emissions", "metrics", "targets", "commercial_banking"],
}

def compile_gap_directives(full_payload):
    ryear = full_payload["metadata"]["reporting_year"]
    directives = []
    for i, gap in enumerate(full_payload["metadata"]["data_gaps"], start=1):
        field = gap["field"]
        d = {
            "directive_id": f"GAP-{i:02d}",
            "field": field,
            "affected_years": gap["affected_years"],
            "reason": gap["reason"],
            "instruction": gap["instruction"],
            "related_tags": GAP_FIELD_TO_TAGS.get(field, []),
            "forbidden_claims": [],
            "required_disclosure_posture": "",
        }
        if field == "scope1_fleet_tco2e":
            d["forbidden_claims"] = [
                f"Any fleet Scope 1 figure (including zero) for years {gap['affected_years']}",
                "Presenting Scope 1 totals for 2022-2023 as like-for-like comparable with 2024 without noting the fleet coverage change",
            ]
            d["required_disclosure_posture"] = (
                "Report fleet emissions for {} only; state prior-year vehicle activity data is not available; "
                "comparative Scope 1 figures cover stationary combustion (gas) only.".format(ryear))
        elif field == "scope3_travel_tco2e":
            d["forbidden_claims"] = [
                f"Any Scope 3 category 6 figure (including zero) for years {gap['affected_years']}",
                "Any year-on-year trend statement for business travel emissions",
            ]
            d["required_disclosure_posture"] = (
                "Report Scope 3 category 6 for {} only; state comparatives are not available.".format(ryear))
        elif field == "investments.counterparty_id":
            d["forbidden_claims"] = [
                "Issuer-level or counterparty-level claims about investment financed emissions",
                "PCAF data quality better than score 3 for investment attribution",
            ]
            d["required_disclosure_posture"] = (
                "Attribute investment financed emissions at portfolio level; disclose PCAF data quality score 3 "
                "for investment attribution.")
        elif field == "total_loans_meur":
            d["forbidden_claims"] = [
                "Any statement describing growth, decline or trend in the loan book / lending volumes",
                "Attributing carbon intensity changes to changes in lending volume",
            ]
            d["required_disclosure_posture"] = (
                "State explicitly that the financed-emissions intensity trend reflects changes in absolute "
                "emissions only, because the loan-book denominator is held constant in the underlying data; "
                "disclose this as a methodology limitation.")
        directives.append(d)
    return directives

## Cell 7 — Computed metrics engine

All arithmetic happens here. Notable gap-aware behaviour:

- Scope 1 comparatives (2022–23) are **gas-only**; the metric carries a caveat forbidding any fleet figure (including zero) for those years. The only permitted like-for-like Scope 1 trend is the stationary-combustion series.
- Scope 3 category 6 exists for 2024 only — no trend metrics are generated for it.
- Financed-emissions intensity metrics carry the constant-denominator caveat (GAP-04): trends reflect emission changes only.
- Target progress uses the payload's `schedule_elapsed` proxies as-is, flagged so they are never presented as emissions progress.
- PCAF data-quality scores are exposure-weighted per asset class.

In [18]:
# ------------------------------------------------------------
# Step 4 - Computed metrics engine (all arithmetic happens HERE, never in the LLM)
# ------------------------------------------------------------

def _metric(mid, name, value, unit, provenance, years=None, decimals=1,
            caveats=None, sections=None):
    return {
        "metric_id": mid,
        "name": name,
        "value": value,
        "display": fmt_num(value, decimals),
        "unit": unit,
        "years": years,
        "provenance": provenance,   # payload paths and/or formula
        "caveats": caveats or [],
        "sections": sections or [],  # which report sections may use it
    }


def compute_metrics(full):
    m = []
    ryear = full["metadata"]["reporting_year"]
    comp_years = full["metadata"]["comparative_years"]
    ALL = ["general_requirements", "governance", "strategy", "risk_management", "metrics_and_targets"]
    MT = ["metrics_and_targets", "general_requirements"]

    def rows(table):
        return sorted(full[table], key=lambda r: r["reporting_year"])

    # --- Scope 1 (gap-aware) ---
    for r in rows("scope1"):
        y = r["reporting_year"]
        cav = []
        if not r["fleet_data_available"]:
            cav = ["Fleet activity data not available for this year; total covers stationary combustion (gas) only. "
                   "Never present a fleet figure (including zero) for this year."]
        m.append(_metric(f"scope1_total_{y}", f"Scope 1 total {y}", r["scope1_total_tco2e"], "tCO2e",
                         {"path": f"scope1[year={y}].scope1_total_tco2e"}, [y], 1, cav, MT + ["strategy"]))
        if r["fleet_data_available"]:
            m.append(_metric(f"scope1_fleet_{y}", f"Scope 1 fleet {y}", r["scope1_fleet_tco2e"], "tCO2e",
                             {"path": f"scope1[year={y}].scope1_fleet_tco2e"}, [y], 1, [], MT))
        m.append(_metric(f"scope1_gas_{y}", f"Scope 1 stationary combustion {y}", r["scope1_gas_tco2e"], "tCO2e",
                         {"path": f"scope1[year={y}].scope1_gas_tco2e"}, [y], 1, [], MT))

    # Gas-only YoY (the only like-for-like Scope 1 comparison permitted)
    s1 = {r["reporting_year"]: r for r in full["scope1"]}
    for y0, y1 in zip(sorted(s1)[:-1], sorted(s1)[1:]):
        delta = (s1[y1]["scope1_gas_tco2e"] - s1[y0]["scope1_gas_tco2e"]) / s1[y0]["scope1_gas_tco2e"] * 100
        m.append(_metric(f"scope1_gas_yoy_{y0}_{y1}", f"Scope 1 gas change {y0}->{y1}", delta, "%",
                         {"formula": f"(gas_{y1}-gas_{y0})/gas_{y0}*100"}, [y0, y1], 1,
                         ["Like-for-like comparison valid for stationary combustion only."], MT))

    # --- Scope 2 ---
    s2 = {r["reporting_year"]: r for r in full["scope2"]}
    for y, r in s2.items():
        m.append(_metric(f"scope2_location_{y}", f"Scope 2 location-based {y}", r["scope2_location_tco2e"], "tCO2e",
                         {"path": f"scope2[year={y}].scope2_location_tco2e"}, [y], 1, [], MT + ["strategy"]))
        m.append(_metric(f"scope2_market_{y}", f"Scope 2 market-based {y}", r["scope2_market_tco2e"], "tCO2e",
                         {"path": f"scope2[year={y}].scope2_market_tco2e"}, [y], 1, [], MT + ["strategy"]))
    for basis in ["location", "market"]:
        ys = sorted(s2)
        for y0, y1 in zip(ys[:-1], ys[1:]):
            a, b = s2[y0][f"scope2_{basis}_tco2e"], s2[y1][f"scope2_{basis}_tco2e"]
            m.append(_metric(f"scope2_{basis}_yoy_{y0}_{y1}", f"Scope 2 {basis}-based change {y0}->{y1}",
                             (b - a) / a * 100, "%", {"formula": f"({basis}_{y1}-{basis}_{y0})/{basis}_{y0}*100"},
                             [y0, y1], 1, [], MT))

    # --- Scope 3 travel (2024 only) ---
    t = full["scope3_travel"][0]
    m.append(_metric(f"scope3_travel_{t['reporting_year']}", "Scope 3 category 6 (business travel)",
                     t["scope3_travel_tco2e"], "tCO2e",
                     {"path": "scope3_travel[year=2024].scope3_travel_tco2e"}, [t["reporting_year"]], 1,
                     ["Comparatives not available (GAP-02). Never state a prior-year travel figure or trend."], MT))

    # --- Total own-operations GHG 2024 (market and location basis) ---
    own_mkt = s1[ryear]["scope1_total_tco2e"] + s2[ryear]["scope2_market_tco2e"] + t["scope3_travel_tco2e"]
    own_loc = s1[ryear]["scope1_total_tco2e"] + s2[ryear]["scope2_location_tco2e"] + t["scope3_travel_tco2e"]
    m.append(_metric(f"own_ops_total_market_{ryear}", "Own operations GHG total (S1 + S2 market + S3 cat.6)",
                     own_mkt, "tCO2e", {"formula": "scope1_total+scope2_market+scope3_travel (2024)"},
                     [ryear], 1, [], MT))
    m.append(_metric(f"own_ops_total_location_{ryear}", "Own operations GHG total (S1 + S2 location + S3 cat.6)",
                     own_loc, "tCO2e", {"formula": "scope1_total+scope2_location+scope3_travel (2024)"},
                     [ryear], 1, [], MT))

    # --- Financed emissions + intensity (denominator caveat from GAP-04) ---
    fe = {r["reporting_year"]: r for r in full["financed_emissions"]}
    denom_caveat = ("Loan-book denominator constant across years in underlying data (GAP-04): intensity trend "
                    "reflects emission changes only; disclose as methodology limitation; never describe loan-book trends.")
    for y, r in fe.items():
        m.append(_metric(f"financed_emissions_loans_{y}", f"Financed emissions - lending {y}",
                         r["financed_em_loans_tco2e"], "tCO2e",
                         {"path": f"financed_emissions[year={y}].financed_em_loans_tco2e"}, [y], 0, [], MT + ["strategy"]))
        m.append(_metric(f"carbon_intensity_lending_{y}", f"Lending carbon intensity {y}",
                         r["carbon_intensity_tco2e_per_meur_lending"], "tCO2e/MEUR",
                         {"path": f"financed_emissions[year={y}].carbon_intensity_tco2e_per_meur_lending"},
                         [y], 1, [denom_caveat], MT + ["strategy"]))
    ys = sorted(fe)
    for y0, y1 in zip(ys[:-1], ys[1:]):
        a, b = fe[y0]["financed_em_loans_tco2e"], fe[y1]["financed_em_loans_tco2e"]
        m.append(_metric(f"financed_emissions_yoy_{y0}_{y1}", f"Financed emissions change {y0}->{y1}",
                         (b - a) / a * 100, "%", {"formula": f"(fe_{y1}-fe_{y0})/fe_{y0}*100"}, [y0, y1], 1,
                         [denom_caveat], MT))

    # --- PCAF weighted data quality scores ---
    for table, label in [("financed_emissions_equity", "equity_investments"),
                         ("financed_emissions_sovereign", "sovereign_bonds")]:
        rs = [r for r in full[table] if r.get("reporting_year", ryear) == ryear] or full[table]
        wsum = sum(r["nominal_amount_meur"] for r in rs)
        dqs = sum(r["pcaf_data_quality_score"] * r["nominal_amount_meur"] for r in rs) / wsum
        m.append(_metric(f"pcaf_dqs_{label}_{ryear}", f"PCAF exposure-weighted DQS - {label}", dqs, "score (1-5)",
                         {"formula": f"sum(dqs*nominal)/sum(nominal) over {table}"}, [ryear], 1,
                         ["Investment attribution at bank level only; PCAF DQS 3 floor applies (GAP-03)."], MT))
        m.append(_metric(f"exposure_{label}_{ryear}", f"Nominal exposure - {label}", wsum, "MEUR",
                         {"formula": f"sum(nominal_amount_meur) over {table}"}, [ryear], 1, [], MT))

    # --- Targets: progress where computable, schedule-elapsed proxies passed through ---
    for ts in full["reporting_kpis"]["target_summary"]:
        tid = ts["target_id"]
        if ts.get("target_progress_pct_2024") is not None:
            m.append(_metric(f"target_progress_{tid}_{ryear}", f"Target {tid} progress", ts["target_progress_pct_2024"],
                             "%", {"path": f"reporting_kpis.target_summary[{tid}].target_progress_pct_2024"},
                             [ryear], 1, [], MT + ["strategy"]))
        if ts.get("schedule_elapsed_pct_2024") is not None:
            m.append(_metric(f"target_schedule_elapsed_{tid}_{ryear}", f"Target {tid} schedule elapsed",
                             ts["schedule_elapsed_pct_2024"], "%",
                             {"path": f"reporting_kpis.target_summary[{tid}].schedule_elapsed_pct_2024"}, [ryear], 1,
                             ["Schedule-elapsed proxy only; never present as emissions progress against the target."],
                             MT + ["strategy"]))
    # Intensity target: measurable progress from baseline (TGT002 baseline 2022 intensity)
    for tg in full["targets"]:
        if tg["metric"] == "tco2e_per_meur_lending" and tg["baseline_year"] in fe:
            base = tg["baseline_value"]
            cur = fe[ryear]["carbon_intensity_tco2e_per_meur_lending"]
            m.append(_metric(f"target_{tg['target_id']}_intensity_change_{ryear}",
                             f"{tg['target_id']} intensity change vs baseline {tg['baseline_year']}",
                             (cur - base) / base * 100, "%",
                             {"formula": f"(intensity_{ryear}-baseline)/baseline*100",
                              "baseline_path": f"targets[{tg['target_id']}].baseline_value"},
                             [tg["baseline_year"], ryear], 1, [denom_caveat], MT + ["strategy"]))

    # --- Governance / board minutes stats ---
    minutes = [b for b in full["board_minutes"] if b["reporting_year"] == ryear]
    for ctype in sorted({b["committee_type"] for b in minutes}):
        ms = [b for b in minutes if b["committee_type"] == ctype]
        clim = [b for b in ms if b["climate_agenda_flag"]]
        dec = [b for b in ms if b["decision_made_flag"]]
        gsec = ["governance", "general_requirements"]
        m.append(_metric(f"meetings_{ctype}_{ryear}", f"{ctype} meetings held {ryear}", len(ms), "count",
                         {"formula": f"count(board_minutes[type={ctype}, year={ryear}])"}, [ryear], 0, [], gsec))
        m.append(_metric(f"meetings_climate_{ctype}_{ryear}", f"{ctype} meetings with climate on agenda {ryear}",
                         len(clim), "count", {"formula": "count(climate_agenda_flag=true)"}, [ryear], 0, [], gsec))
        m.append(_metric(f"meetings_climate_pct_{ctype}_{ryear}", f"{ctype} climate agenda share {ryear}",
                         len(clim) / len(ms) * 100 if ms else None, "%",
                         {"formula": "climate_meetings/total_meetings*100"}, [ryear], 0, [], gsec))
        m.append(_metric(f"decisions_{ctype}_{ryear}", f"{ctype} climate-related decisions {ryear}", len(dec),
                         "count", {"formula": "count(decision_made_flag=true)"}, [ryear], 0, [], gsec))

    # --- Risk register / physical risk aggregates ---
    reg = [r for r in full["climate_risk_register"] if r["reporting_year"] == ryear] or full["climate_risk_register"]
    m.append(_metric(f"risks_registered_{ryear}", "Climate risks in register", len(reg), "count",
                     {"formula": "count(climate_risk_register)"}, [ryear], 0, [],
                     ["risk_management", "strategy", "general_requirements"]))
    for cat in sorted({r["risk_category"] for r in reg}):
        n = len([r for r in reg if r["risk_category"] == cat])
        m.append(_metric(f"risks_{cat}_{ryear}", f"Risks - {cat}", n, "count",
                         {"formula": f"count(risk_category={cat})"}, [ryear], 0, [], ["risk_management"]))
    phys = full.get("physical_risk_exposures", [])
    if phys:
        hi = [p for p in phys if p.get("high_risk_flag")]
        m.append(_metric(f"physical_high_risk_exposure_{ryear}", "High-physical-risk exposure",
                         sum(p["exposure_amount_meur"] for p in hi), "MEUR",
                         {"formula": "sum(exposure_amount_meur where high_risk_flag)"}, [ryear], 1, [],
                         ["risk_management", "strategy"]))
        m.append(_metric(f"physical_exposures_assessed_{ryear}", "Counterparty physical-risk assessments",
                         len(phys), "count", {"formula": "count(physical_risk_exposures)"}, [ryear], 0, [],
                         ["risk_management"]))

    # --- Scope 3 categories screen ---
    cats = [c for c in full["scope3_categories"] if c["reporting_year"] == ryear]
    inc = [c for c in cats if c["included_flag"]]
    m.append(_metric(f"scope3_categories_included_{ryear}", "Scope 3 categories included", len(inc), "count",
                     {"formula": "count(included_flag=true, 2024)"}, [ryear], 0, [], MT))
    m.append(_metric(f"scope3_categories_excluded_{ryear}", "Scope 3 categories excluded", len(cats) - len(inc),
                     "count", {"formula": "count(included_flag=false, 2024)"}, [ryear], 0, [], MT))
    m.append(_metric(f"scope3_included_total_{ryear}", "Scope 3 included categories total",
                     sum(c["emissions_tco2e"] or 0 for c in inc), "tCO2e",
                     {"formula": "sum(emissions_tco2e over included categories, 2024)"}, [ryear], 0, [], MT))

    # --- Carbon credits ---
    cc = [c for c in full["carbon_credits"] if c["reporting_year"] == ryear] or full["carbon_credits"]
    ret = [c for c in cc if c.get("retirement_year")]
    m.append(_metric(f"carbon_credits_tonnes_{ryear}", "Carbon credits volume", sum(c["tonnes_co2e"] for c in cc),
                     "tCO2e", {"formula": "sum(tonnes_co2e)"}, [ryear], 0,
                     ["Distinguish planned vs retired credits per record 'use'/'planned_flag' fields."], MT))
    m.append(_metric(f"carbon_credits_retired_tonnes_{ryear}", "Carbon credits retired",
                     sum(c["tonnes_co2e"] for c in ret), "tCO2e", {"formula": "sum where retirement_year set"},
                     [ryear], 0, [], MT))

    # --- Financial / exposure KPIs passthrough (formatted centrally for consistency) ---
    kpi = full["reporting_kpis"]
    passthrough = [
        ("green_loans_pct_2024", "Green loans share", "%", 1, MT + ["strategy"]),
        ("climate_capex_2024_meur", "Climate-related capex", "MEUR", 1, MT + ["strategy"]),
        ("climate_opex_2024_meur", "Climate-related opex", "MEUR", 1, MT + ["strategy"]),
        ("total_assets_2024_meur", "Total assets", "MEUR", 0, ALL),
        ("total_loans_2024_meur", "Total loans", "MEUR", 0, ALL),
        ("high_carbon_sector_exposure_pct", "High-carbon sector exposure", "%", 1, ["strategy", "risk_management", "metrics_and_targets"]),
        ("fossil_fuel_exposure_pct", "Fossil fuel exposure", "%", 1, ["strategy", "risk_management", "metrics_and_targets"]),
        ("high_carbon_sector_exposure_meur", "High-carbon sector exposure (MEUR)", "MEUR", 1, ["strategy", "risk_management", "metrics_and_targets"]),
        ("fossil_fuel_exposure_meur", "Fossil fuel exposure (MEUR)", "MEUR", 1, ["strategy", "risk_management", "metrics_and_targets"]),
    ]
    for k, name, unit, dec, secs in passthrough:
        if kpi.get(k) is not None:
            m.append(_metric(f"kpi_{k}", name, kpi[k], unit, {"path": f"reporting_kpis.{k}"}, [ryear], dec, [], secs))

    return {x["metric_id"]: x for x in m}

## Cell 8 — Allowed numbers & entity allowlist

`allowed_numbers` is every numeric value in the payload and computed metrics, expanded into display variants (thousands separators, 0–2 dp, MtCO2e/kt scalings). Phase B's numeric gate rejects any numeral in a draft that is not in this set. `entity_allowlist` collects named entities (committees, scenarios, frameworks, providers, issuers) — the fact judge flags any name outside it.

In [19]:
# ------------------------------------------------------------
# Step 5 - Allowed numbers + entity allowlist (for downstream hard gates)
# ------------------------------------------------------------

def _num_variants(v):
    out = set()
    if v is None or isinstance(v, bool):
        return out
    try:
        f = float(v)
    except (TypeError, ValueError):
        return out
    for dec in (0, 1, 2):
        out.add(f"{f:,.{dec}f}")
        out.add(f"{f:.{dec}f}")
    if f == int(f):
        out.add(str(int(f)))
    out.add(str(v))
    # thousands-scaled variants (e.g. 35,973,168 tCO2e ~ 36.0 MtCO2e)
    if abs(f) >= 1_000_000:
        out.add(f"{f/1_000_000:,.1f}")
        out.add(f"{f/1_000_000:,.2f}")
    if abs(f) >= 1_000:
        out.add(f"{f/1_000:,.1f}")
    return out


def collect_allowed_numbers(obj, acc=None):
    if acc is None:
        acc = set()
    if isinstance(obj, dict):
        for v in obj.values():
            collect_allowed_numbers(v, acc)
    elif isinstance(obj, list):
        for v in obj:
            collect_allowed_numbers(v, acc)
    elif isinstance(obj, (int, float)) and not isinstance(obj, bool):
        acc |= _num_variants(obj)
    return acc


ENTITY_STRING_KEYS = {
    "bank_name", "assurance_provider", "committee_name", "management_committee_name",
    "scenario_name", "framework", "target_framework", "validation_body", "registry",
    "issuer_name", "standards_basis", "regulatory_regime", "benchmark_reference",
    "aligned_framework", "country", "reporting_entity",
}

def collect_entity_allowlist(obj, acc=None):
    if acc is None:
        acc = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in ENTITY_STRING_KEYS and isinstance(v, str) and v.strip():
                acc.add(v.strip())
            else:
                collect_entity_allowlist(v, acc)
    elif isinstance(obj, list):
        for v in obj:
            collect_entity_allowlist(v, acc)
    return acc


# ------------------------------------------------------------
# Step 6 - Requirement enrichment (mapping + gap attachment + availability)
# ------------------------------------------------------------

## Cell 9 — Requirement enrichment

Maps each requirement to available payload keys (paragraph overrides → tags → section defaults), attaches relevant gap directives by tag overlap, and classifies availability. `unmapped` mandatory requirements are a build failure.

In [20]:
def enrich_requirement(req, payload_slice, gap_directives, bank_archetype, section_key):
    tags = req.get("evidence_tags") or []
    std, pid = req["standard"], req["paragraph_id"]

    mapped = list(PARAGRAPH_OVERRIDES.get((std, pid), []))
    for tg in tags:
        mapped += TAG_TO_PAYLOAD_KEYS.get(tg, [])
    if not mapped and not tags:
        mapped = list(SECTION_DEFAULT_KEYS.get(section_key, []))
    # de-dupe, preserve order
    seen, ordered = set(), []
    for k in mapped:
        if k not in seen:
            seen.add(k)
            ordered.append(k)

    available, unavailable = [], []
    for k in ordered:
        top = k.split(".")[0]
        if top in payload_slice and (get_by_path(payload_slice, k) is not None):
            available.append(k)
        else:
            unavailable.append(k)

    gaps = [g["directive_id"] for g in gap_directives if set(g["related_tags"]) & set(tags)]

    handling = PARAGRAPH_HANDLING.get((std, pid), "disclosure")

    if handling in ("drafting_constraint", "fixed_block", "conditional_event"):
        status = "presentation_rule"
    elif any(t in CONDITIONAL_TAGS for t in tags) and not available:
        status = "not_applicable_archetype" if "insurance" in tags else "conditional_check_archetype"
    elif available:
        status = "data_backed"
    elif set(tags) & NARRATIVE_OK_TAGS or not tags:
        status = "narrative_only"
    else:
        status = "unmapped"  # must be resolved before generation

    return {
        "requirement_id": req["requirement_id"],
        "standard": std,
        "paragraph_id": pid,
        "clause_path": req.get("clause_path"),
        "mandatory": bool(req.get("mandatory")),
        "obligation_type": req.get("obligation_type"),
        "banking_relevance": req.get("banking_relevance"),
        "evidence_tags": tags,
        "requirement_text": req.get("clean_requirement_text") or req["requirement_text"],
        "mapped_payload_keys": available,
        "unavailable_payload_keys": unavailable,
        "gap_directive_ids": gaps,
        "availability_status": status,
        "handling_hint": handling,
    }

## Cell 10 — Work-package assembly & export

In [21]:
# ------------------------------------------------------------
# Step 7 - Work package assembly + export
# ------------------------------------------------------------

def build_work_packages(base_dir: Path, out_dir: Path, style_dir: Path = None, payload_dir: Path = None):
    payload_dir = payload_dir or base_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    report = {"problems": [], "sections": {}}

    kbs, p1 = load_requirements(base_dir)
    full, slices, p2 = load_payloads(payload_dir)
    report["problems"] += p1 + p2

    gap_directives = compile_gap_directives(full)
    metrics = compute_metrics(full)
    archetype = full["bank"]["archetype"]

    global_numbers = collect_allowed_numbers(full)
    for mt in metrics.values():
        global_numbers |= _num_variants(mt["value"])
    # years and simple context numbers
    for y in [full["metadata"]["reporting_year"]] + full["metadata"]["comparative_years"]:
        global_numbers.add(str(y))
    entities = sorted(collect_entity_allowlist(full))

    manifest = {
        "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "bank_id": full["metadata"]["bank_id"],
        "reporting_year": full["metadata"]["reporting_year"],
        "input_hashes": {},
        "outputs": [],
    }

    for key, cfg in SECTION_REGISTRY.items():
        if key not in kbs:
            continue
        sl = slices[key]
        enriched = [enrich_requirement(r, sl, gap_directives, archetype, key) for r in kbs[key]["requirements"]]

        sec_metrics = {mid: mt for mid, mt in metrics.items() if key in mt["sections"]}
        sec_gap_ids = sorted({g for e in enriched for g in e["gap_directive_ids"]})
        sec_gaps = [g for g in gap_directives if g["directive_id"] in sec_gap_ids] or gap_directives

        style_refs = {}
        if style_dir is not None:
            style_refs = {
                "global_style_guide": str(style_dir / "global_style_guide.json"),
                "section_blueprint": str(style_dir / "authoring" / "section_blueprints" / f"{key}_blueprint.json"),
                "section_style_guide": str(style_dir / "authoring" / "section_style_guides" / f"{key}_style_guide.json"),
                "table_patterns_dir": str(style_dir / "authoring" / "table_patterns"),
                "style_rubric": str(style_dir / "judging" / "style_compliance_rubric.json"),
            }

        wp = {
            "work_package_version": "1.0",
            "section_key": key,
            "section_title": cfg["title"],
            "bank_id": full["metadata"]["bank_id"],
            "reporting_year": full["metadata"]["reporting_year"],
            "comparative_years": full["metadata"]["comparative_years"],
            "requirements": enriched,
            "payload_slice": sl,
            "computed_metrics": sec_metrics,
            "gap_directives": sec_gaps,
            "entity_allowlist": entities,
            "style_artifacts": style_refs,
            "counts": {
                "requirements_total": len(enriched),
                "mandatory": sum(1 for e in enriched if e["mandatory"]),
                "data_backed": sum(1 for e in enriched if e["availability_status"] == "data_backed"),
                "narrative_only": sum(1 for e in enriched if e["availability_status"] == "narrative_only"),
                "presentation_rule": sum(1 for e in enriched if e["availability_status"] == "presentation_rule"),
                "not_applicable_archetype": sum(1 for e in enriched if e["availability_status"] == "not_applicable_archetype"),
                "conditional": sum(1 for e in enriched if e["availability_status"] == "conditional_check_archetype"),
                "unmapped": sum(1 for e in enriched if e["availability_status"] == "unmapped"),
            },
        }
        out_path = out_dir / f"work_package_{key}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(wp, f, indent=1, ensure_ascii=False)
        manifest["outputs"].append(str(out_path))
        report["sections"][key] = wp["counts"]

        unmapped = [e["requirement_id"] for e in enriched
                    if e["availability_status"] == "unmapped" and e["mandatory"]]
        if unmapped:
            report["problems"].append(f"[{key}] mandatory unmapped requirements: {unmapped}")

    # shared artifacts
    with open(out_dir / "allowed_numbers.json", "w") as f:
        json.dump(sorted(global_numbers), f, indent=0)
    with open(out_dir / "entity_allowlist.json", "w") as f:
        json.dump(entities, f, indent=1, ensure_ascii=False)
    with open(out_dir / "gap_directives.json", "w") as f:
        json.dump(gap_directives, f, indent=1, ensure_ascii=False)
    with open(out_dir / "computed_metrics_all.json", "w") as f:
        json.dump(metrics, f, indent=1, ensure_ascii=False)

    for key, cfg in SECTION_REGISTRY.items():
        for fkey in ["requirements_file", "payload_file"]:
            p = (base_dir if fkey == "requirements_file" else payload_dir) / cfg[fkey]
            if p.exists():
                manifest["input_hashes"][cfg[fkey]] = sha256_file(p)
    manifest["input_hashes"]["payload_BANK01.json"] = sha256_file(payload_dir / "payload_BANK01.json")
    with open(out_dir / "run_manifest.json", "w") as f:
        json.dump(manifest, f, indent=1)

    return report, manifest

## Cell 11 — Run + validation gates

The summary must show **zero `unmapped`** and no problems before Phase B may run.

In [22]:
# ============================================================
# CELL 9 — RUN PHASE A + VALIDATION REPORT
# ============================================================

report, manifest = build_work_packages(
    base_dir=REQUIREMENTS_DIR,          # requirement KBs
    out_dir=OUTPUT_DIR,
    style_dir=STYLE_SYSTEM_DIR,
    payload_dir=PAYLOAD_DIR,            # payload slices (may equal REQUIREMENTS_DIR)
)

summary_df = pd.DataFrame(report["sections"]).T
display(summary_df)

print()
if report["problems"]:
    for p in report["problems"]:
        print("PROBLEM:", p)
else:
    print("No validation problems.")

# Hard gates: Phase B must not start unless these hold.
assert not report["problems"], "Phase A validation failed — fix problems above before generation."
assert (summary_df["unmapped"] == 0).all(), "Unmapped mandatory requirements remain."
assert (summary_df["requirements_total"] == summary_df["mandatory"]).all() or True  # informational

total_reqs = int(summary_df["requirements_total"].sum())
print(f"\nPhase A complete: {total_reqs} requirements packaged across {len(summary_df)} sections.")
print("Outputs:")
for o in manifest["outputs"]:
    print("  -", o)

,requirements_total,mandatory,data_backed,narrative_only,presentation_rule,not_applicable_archetype,conditional,unmapped
general_requirements,108,108,97,0,11,0,0,0
governance,15,15,14,0,1,0,0,0
strategy,70,70,69,1,0,0,0,0
risk_management,17,17,16,0,1,0,0,0
metrics_and_targets,151,151,149,0,0,2,0,0



No validation problems.

Phase A complete: 361 requirements packaged across 5 sections.
Outputs:
  - c:\Users\HP\Documents\IFRS_Reporting\notebooks\gen_data\generation\phase_a\work_package_general_requirements.json
  - c:\Users\HP\Documents\IFRS_Reporting\notebooks\gen_data\generation\phase_a\work_package_governance.json
  - c:\Users\HP\Documents\IFRS_Reporting\notebooks\gen_data\generation\phase_a\work_package_strategy.json
  - c:\Users\HP\Documents\IFRS_Reporting\notebooks\gen_data\generation\phase_a\work_package_risk_management.json
  - c:\Users\HP\Documents\IFRS_Reporting\notebooks\gen_data\generation\phase_a\work_package_metrics_and_targets.json
